# Plant Disease Classification

This notebook walks through PlantVillage acquisition, exploratory analysis, a reproducible train/validation/test split, MobileNetV2 transfer learning, held-out evaluation, and one test-image prediction. The dataset is capped at 200 images per class. Keep the test split untouched when training or retraining.

Run cells from the project root with the project `.venv` active. First execution downloads the PlantVillage source archive and may take several minutes.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from src.data_acquisition import acquire_plant_village_dataset
from src.preprocessing import create_split_manifest, load_split_datasets
from src.visualization import generate_dataset_visualizations
from src.training import train_and_evaluate
from src.prediction import load_predictor

PROJECT_DIRECTORY = Path.cwd().resolve()
if not (PROJECT_DIRECTORY / 'src').is_dir():
    raise FileNotFoundError('Run this notebook from the project root.')
DATASET_METADATA = PROJECT_DIRECTORY / 'data' / 'raw' / 'dataset_metadata.json'
RANDOM_SEED = 42

## 1. Acquire and inspect the subset

The acquisition script exports up to 200 images for each of the 38 disease/healthy classes. It records class counts, seed, and source metadata. If the dataset is already present, this cell reuses it instead of overwriting files.

In [ ]:
if not DATASET_METADATA.is_file():
    acquisition_report = acquire_plant_village_dataset(
        PROJECT_DIRECTORY,
        max_images_per_class=200,
        random_seed=RANDOM_SEED
    )
    print(acquisition_report)
else:
    print('Using existing dataset metadata:', DATASET_METADATA)

In [ ]:
with DATASET_METADATA.open(encoding='utf-8') as file:
    dataset_metadata = json.load(file)

class_counts = pd.Series(dataset_metadata['images_per_class']).sort_values()
print(f"Classes: {dataset_metadata['number_of_classes']}")
print(f"Images: {dataset_metadata['total_images']}")
display(class_counts.to_frame('image_count'))

ax = class_counts.plot(kind='barh', figsize=(10, 12), color='forestgreen')
ax.set_title('PlantVillage subset: images per class')
ax.set_xlabel('Images')
plt.tight_layout()
plt.show()

## 2. Image characteristics and preprocessing

The project resizes images to 224 × 224 RGB pixels and applies MobileNetV2's `preprocess_input`, which scales pixel values to the range expected by its pretrained feature extractor. A fixed-seed, stratified 70/15/15 split keeps every class represented in train, validation, and test. The manifest assigns paths to splits without duplicating image files. Augmentation is deliberately kept out of validation and test.

In [ ]:
split_report = create_split_manifest(
    PROJECT_DIRECTORY,
    train_fraction=0.70,
    validation_fraction=0.15,
    test_fraction=0.15,
    random_seed=RANDOM_SEED
)
print('Images per split:', split_report['images_per_split'])

datasets = load_split_datasets(
    PROJECT_DIRECTORY,
    image_size=(224, 224),
    batch_size=32,
    random_seed=RANDOM_SEED
)
images, labels = next(iter(datasets['train']))
print('Training image batch:', images.shape)
print('Training label batch:', labels.shape)
print('Preprocessed pixel range:', float(images.numpy().min()), float(images.numpy().max()))

In [ ]:
visualization_report = generate_dataset_visualizations(
    PROJECT_DIRECTORY,
    random_seed=RANDOM_SEED
)
print(visualization_report)
print((PROJECT_DIRECTORY / 'visualizations' / 'feature_interpretations.txt').read_text(encoding='utf-8'))

The visual summary compares mean brightness, green-channel ratio, and edge contrast between healthy and diseased classes. These are measurable image descriptors, not proof of causation or explanations of the neural network's internal reasoning.

## 3. Train with transfer learning

MobileNetV2 starts with ImageNet weights. First, its pretrained feature layers are frozen while a regularized softmax classification head is trained with Adam. Then the last eligible convolutional layers are fine-tuned at a smaller learning rate. Early stopping monitors validation loss and restores the best weights. Set `RUN_TRAINING` to `True` to train again; this takes time on CPU.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    training_report = train_and_evaluate(
        PROJECT_DIRECTORY,
        head_epochs=5,
        fine_tune_epochs=3,
        batch_size=32,
        fine_tune_layers=20,
        random_seed=RANDOM_SEED,
        weights='imagenet'
    )
    print({key: value for key, value in training_report.items() if key != 'classification_report'})
else:
    print('Training skipped. Set RUN_TRAINING=True to train a new model.')

## 4. Evaluate the held-out test set

The test split is used only for the final report, not fitting or retraining. The saved report includes test loss, accuracy, macro/weighted precision, recall and F1, per-class results, and the confusion matrix.

In [ ]:
metrics_path = PROJECT_DIRECTORY / 'models' / 'evaluation_metrics.json'
with metrics_path.open(encoding='utf-8') as file:
    metrics = json.load(file)
pd.Series({
    'Test loss': metrics['test_loss'],
    'Accuracy': metrics['accuracy'],
    'Macro precision': metrics['macro_precision'],
    'Macro recall': metrics['macro_recall'],
    'Macro F1': metrics['macro_f1'],
    'Weighted precision': metrics['weighted_precision'],
    'Weighted recall': metrics['weighted_recall'],
    'Weighted F1': metrics['weighted_f1']
}).to_frame('value')

## 5. Predict one held-out image

This example selects an image from the test split and submits it to the same prediction preprocessing used by the API. Its ground-truth label is shown beside the top predictions.

In [ ]:
import csv

manifest_path = PROJECT_DIRECTORY / 'data' / 'processed' / 'split_manifest.csv'
with manifest_path.open(newline='', encoding='utf-8') as file:
    test_rows = [row for row in csv.DictReader(file) if row['split'] == 'test']
sample = test_rows[0]
sample_path = PROJECT_DIRECTORY / sample['relative_path']
display(Image.open(sample_path))

predictor = load_predictor(PROJECT_DIRECTORY)
predictions = predictor.predict(sample_path.read_bytes(), top_k=3)
print('Ground truth:', sample['class_name'])
display(pd.DataFrame(predictions))